Import libraries

In [15]:
import json
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"D:\FactoryMindAI")
PROCESSED_PATH = PROJECT_ROOT / "knowledge_base" / "processed"

chunks_file = PROCESSED_PATH / "chunked_documents.jsonl"
embeddings_file = PROCESSED_PATH / "chunk_embeddings.npy"

print("Chunks file exists:", chunks_file.exists())
print("Embeddings file exists:", embeddings_file.exists())

embeddings = np.load(embeddings_file)

with open(chunks_file, "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

print("Total chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)

Chunks file exists: True
Embeddings file exists: True
Total chunks: 1017
Embedding shape: (1017, 384)


In [16]:
assert len(chunks) == len(embeddings), "Chunk and embedding counts do not match!"
assert embeddings.shape[1] == 384, "Incorrect embedding dimension!"

print("Data validation successful!")

Data validation successful!


In [17]:
from qdrant_client.models import PointStruct

points = []

for index, chunk in enumerate(chunks):

    payload = {
        "chunk_id": chunk["chunk_id"],
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk_index": chunk["chunk_index"],
        "text": chunk["text"]
    }

    point = PointStruct(
        id=index,
        vector=embeddings[index].tolist(),
        payload=payload
    )

    points.append(point)

client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True
)

print("Vectors uploaded successfully!")

NameError: name 'client' is not defined

In [ ]:
collection_info = client.get_collection(COLLECTION_NAME)

print("Total stored vectors:", collection_info.points_count)

NameError: name 'client' is not defined

In [18]:
import json
import numpy as np

from pathlib import Path
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


Connect to your existing Qdrant database

In [19]:
from pathlib import Path
from qdrant_client import QdrantClient

PROJECT_ROOT = Path.cwd().parent

QDRANT_PATH = PROJECT_ROOT / "qdrant_storage"
COLLECTION_NAME = "factorymind_knowledge"

client = QdrantClient(path=str(QDRANT_PATH))

print("Qdrant connected successfully!")
print("Collection exists:", client.collection_exists(COLLECTION_NAME))

Qdrant connected successfully!
Collection exists: True


In [20]:
collection_info = client.get_collection(COLLECTION_NAME)

print("Total stored vectors:", collection_info.points_count)
print("Vector configuration:", collection_info.config.params.vectors)

Total stored vectors: 1017
Vector configuration: size=384 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None memory=None datatype=None multivector_config=None


Load the embedding model

In [21]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


Create the semantic search function

In [22]:
def semantic_search(query, top_k=3):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    search_results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding.tolist(),
        limit=top_k,
        with_payload=True
    ).points

    return search_results

Test your retrieval system

In [23]:
query = "What are the important industrial maintenance practices?"

results = semantic_search(query, top_k=3)

print("Question:", query)
print("Retrieved chunks:", len(results))

Question: What are the important industrial maintenance practices?
Retrieved chunks: 3


Display retrieved documents

In [24]:
for i, result in enumerate(results, start=1):

    payload = result.payload

    print(f"\n========== RESULT {i} ==========")
    print("Similarity Score:", round(result.score, 4))
    print("Source:", payload["source"])
    print("Page:", payload["page"])
    print("Chunk ID:", payload["chunk_id"])

    print("\nRetrieved Text:")
    print(payload["text"])


========== RESULT 1 ==========
Similarity Score: 0.6873
Source: omguide_complete_w-eo-disclaimer.pdf
Page: 49
Chunk ID: omguide_complete_w-eo-disclaimer.pdf_page_49_chunk_0

Retrieved Text:
Chapter 5 Types of Maintenance Programs 
5.1 Introduction
What is maintenance and why is it performed? Past and current maintenance practices in
both the private and government sectors would imply that maintenance is the actions associated
with equipment repair after it is broken. The dictionary defines maintenance as follows: “the work
of keeping something in proper condition; upkeep.” This would imply that maintenance should
be actions taken to prevent a device or component from failing or to repair normal equipment
degradation experienced with the operation of the device to keep it in proper working order.
Unfortunately, data obtained in many studies over the past decade indicates that most private and
government facilities do not expend the necessary resources to maintain equipment in proper w

In [25]:
query = "What are the important industrial maintenance practices?"

results = semantic_search(query, top_k=3)

print("Question:", query)
print("Retrieved chunks:", len(results))

Question: What are the important industrial maintenance practices?
Retrieved chunks: 3


Connect Groq in your notebook

In [26]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=1024
)

print("Groq LLM initialized successfully!")

Groq LLM initialized successfully!


Test your LLM

In [27]:
response = llm.invoke(
    "Explain predictive maintenance in simple words."
)

print(response.content)

**Predictive maintenance** is a way to keep machines running smoothly by spotting problems *before* they actually happen.

Think of it like this:

- **Sensors** on a machine collect data (temperature, vibration, sound, etc.).
- **Software** looks at that data and compares it to what “normal” looks like.
- If the data starts to drift toward a pattern that usually means a part is wearing out or a fault is developing, the system raises a warning.
- You can then replace or repair that part right away, before it breaks and causes a costly shutdown.

So instead of fixing things only after they fail (reactive) or on a fixed schedule that might be too early or too late (preventive), predictive maintenance uses real‑time information to schedule maintenance exactly when it’s needed. This saves money, reduces downtime, and keeps equipment running longer.


Test the Groq LLM

In [28]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    max_tokens=1024
)

response = llm.invoke(
    "Explain predictive maintenance in simple words."
)

print(response.content)

**Predictive maintenance** is a way to keep machines and equipment running smoothly by using data to *predict* when something might break or need service.

### How it works (in plain terms)

1. **Sensors & data**  
   Machines are fitted with simple sensors (vibration, temperature, sound, etc.) that continuously record how the machine is behaving.

2. **Collect & analyze**  
   The data is sent to a computer that looks for patterns. If a machine’s vibration pattern starts to change, that might be a sign that a bearing is wearing out.

3. **Predict a failure**  
   Using math or machine‑learning models, the computer estimates when the problem will become serious enough to cause a breakdown.

4. **Schedule maintenance**  
   Technicians are notified *before* the machine stops. They can replace the part or do a quick fix at a convenient time, avoiding an unexpected shutdown.

### Why it matters

- **Less downtime** – Machines don’t stop suddenly; maintenance happens on schedule.
- **Cost 

Build the RAG answer-generation function

In [29]:
def rag_answer(question, top_k=3):

    results = semantic_search(question, top_k=top_k)

    if not results:
        return "No relevant information found."

    context = ""

    for result in results:
        payload = result.payload

        context += f"""
        Source: {payload['source']}
        Page: {payload['page']}
        Content: {payload['text']}
        """

    prompt = f"""
    You are FactoryMind AI, an industrial maintenance assistant.

    Answer using ONLY the provided context.
    Do not invent information.
    If the answer is unavailable, say so.
    Include source filename and page number.

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    return response.content

Test your RAG system

question = "What are the important industrial maintenance practices?"

answer = rag_answer(question)

print(answer)

In [30]:
import httpx

try:
    response = httpx.get(
        
        "https://api.groq.com",
        timeout=20
    )

    print("Connection successful!")
    print("Status code:", response.status_code)

except Exception as e:
    print("Connection failed!")
    print("Error:", repr(e))

Connection successful!
Status code: 200


In [31]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    print("API key not found!")
else:
    try:
        response = httpx.get(
            "https://api.groq.com/openai/v1/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=20
        )

        print("Status code:", response.status_code)

        if response.status_code == 200:
            print("Groq API connected successfully!")
        else:
            print("API response:", response.text[:500])

    except Exception as e:
        print("Connection error:", repr(e))

Status code: 200
Groq API connected successfully!


Connect Qdrant + Groq

In [40]:
def rag_answer(question, top_k=3, threshold=0.25):

    # 1. Retrieve relevant chunks
    results = semantic_search(question, top_k=top_k)

    if not results:
        return {
            "answer": "No relevant information found in the maintenance documents.",
            "sources": []
        }

    # 2. Filter by relevance score
    relevant_results = [
        result for result in results
        if result.score >= threshold
    ]

    if not relevant_results:
        return {
            "answer": "I couldn't find sufficiently relevant information in the maintenance documents.",
            "sources": []
        }

    # 3. Prepare context and sources
    context_parts = []
    sources = []

    for i, result in enumerate(relevant_results, start=1):

        payload = result.payload

        sources.append({
            "id": f"S{i}",
            "source": payload["source"],
            "page": payload["page"],
            "score": float(result.score)
        })

        context_parts.append(
            f"""
            [S{i}]
            Source: {payload['source']}
            Page: {payload['page']}
            Content: {payload['text']}
            """
        )

    context = "\n\n".join(context_parts)

    # 4. Grounded prompt
    prompt = f"""
    You are FactoryMind AI, an industrial maintenance assistant.

    Answer using ONLY the supplied context.

    Rules:
    - Do not invent facts.
    - Cite factual claims using [S1], [S2], etc.
    - Only cite supplied source labels.
    - If the answer is unavailable, say so.
    - Use simple and clear language.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    # 5. Generate answer
    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "sources": sources
    }

In [33]:
def semantic_search(query, top_k=3):

    # Convert query into embedding
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    # Search in Qdrant
    search_results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding.tolist(),
        limit=top_k,
        with_payload=True
    ).points

    return search_results

In [34]:
results = semantic_search(
    "What are the important industrial maintenance practices?"
)

for result in results:
    print("Score:", result.score)
    print("Page:", result.payload["page"])
    print("Text:", result.payload["text"][:300])
    print("-" * 50)

Score: 0.6872904841925715
Page: 49
Text: Chapter 5 Types of Maintenance Programs 
5.1 Introduction
What is maintenance and why is it performed? Past and current maintenance practices in
both the private and government sectors would imply that maintenance is the actions associated
with equipment repair after it is broken. The dictionary de
--------------------------------------------------
Score: 0.6796496676019674
Page: 50
Text: Types of Maintenance Programs
that exceptional maintenance practices encompassing preventive and predictive elements can extend
this period. The wear-out period is characterized by a rapid increasing failure rate with time. In most
cases this period encompasses the normal distribution of design life
--------------------------------------------------
Score: 0.6556641241181864
Page: 55
Text: Types of Maintenance Programs
3. Assign components into logical groupings.
4. Determine the type and number of maintenance activities required and periodicity using:
a. Manu

Test your complete RAG pipeline

In [35]:
question = "What are the important industrial maintenance practices?"

result = rag_answer(question)

print("FACTORYMIND AI ANSWER:")
print(result["answer"])

print("\nSOURCE DOCUMENTS:")

for source in result["sources"]:
    print(source)

FACTORYMIND AI ANSWER:
**Important industrial maintenance practices (as described in the guide)**  

| Practice | What it involves | Source |
|----------|------------------|--------|
| **Preventive & predictive maintenance** | Regularly scheduled actions (e.g., lubrication, alignment, belt adjustment) that keep equipment in proper condition and extend its design life. | omguide_complete_w-eo-disclaimer.pdf, pp. 49‑50 |
| **Use of manufacturer technical manuals** | Reference the equipment’s design‑specific maintenance requirements. | omguide_complete_w-eo-disclaimer.pdf, pp. 55 |
| **Machinery history review** | Examine past failures and repairs to inform future maintenance planning. | omguide_complete_w-eo-disclaimer.pdf, pp. 55 |
| **Root‑cause analysis** | Identify why equipment failed to prevent recurrence. | omguide_complete_w-eo-disclaimer.pdf, pp. 55 |
| **Good engineering judgment** | Apply expertise to decide on maintenance activities when data is incomplete. | omguide_complete

Test multiple questions

In [36]:
test_questions = [
    "What is preventive maintenance?",
    "Why is machinery history important?",
    "What is root cause analysis?",
    "How can equipment failures be reduced?"
]

for question in test_questions:

    print("\n" + "=" * 70)
    print("QUESTION:", question)

    result = rag_answer(question)

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")
    for source in result["sources"]:
        print(source)

    print("=" * 70)


QUESTION: What is preventive maintenance?

ANSWER:
**Preventive maintenance** is a maintenance approach that performs scheduled actions—based on time or machine run‑hours—to detect, preclude, or mitigate the degradation of a component or system. The goal is to sustain or extend the equipment’s useful life by keeping degradation at an acceptable level, thereby improving reliability and reducing costs compared with a purely reactive program【omguide_complete_w-eo-disclaimer.pdf, Page 51】.

SOURCES:
{'source': 'omguide_complete_w-eo-disclaimer.pdf', 'page': 51, 'score': 0.7068822070084066}
{'source': 'omguide_complete_w-eo-disclaimer.pdf', 'page': 51, 'score': 0.7017281614717278}
{'source': 'omguide_complete_w-eo-disclaimer.pdf', 'page': 51, 'score': 0.6884995158349395}

QUESTION: Why is machinery history important?

ANSWER:
**Answer**

Machinery history is important because it allows maintenance teams to **track and trend machine conditions over time**. By recording data such as motor cu

In [38]:
question = "Why is machinery history important?"

result = rag_answer(question)

print("FACTORYMIND AI ANSWER:")
print(result["answer"])

print("\nVERIFIED RETRIEVED SOURCES:")

for source in result["sources"]:
    print(
        source["id"],
        "| Page:", source["page"],
        "| Score:", round(source["score"], 4)
    )

FACTORYMIND AI ANSWER:
Machinery history matters because it gives you a record of how a machine has behaved over time.  
* By looking at past motor‑current or vibration data you can **trend changes** and spot early signs that the machine is starting to wear out or a process is shifting. This early warning lets you act before a major failure occurs [S1].  
* The history of diagnostics (temperature, vibration, etc.) lets technicians **compare current signatures to known failure modes**, helping them identify what went wrong and why [S2].  
* With a clear history you can **schedule repairs in an orderly way**, order spare parts just in time, and keep the inventory low, all of which boost production capacity [S3].

So, keeping a detailed record of a machine’s past performance helps prevent failures, plan maintenance, and keep the plant running smoothly.

VERIFIED RETRIEVED SOURCES:
S1 | Page: 89 | Score: 0.363
S2 | Page: 237 | Score: 0.3568
S3 | Page: 56 | Score: 0.3568


Test both relevant and irrelevant questions

In [41]:
questions = [
    "What is preventive maintenance?",
    "Why is machinery history important?",
    "Who invented the first computer?"
]

for question in questions:

    result = rag_answer(question)

    print("\nQUESTION:", question)
    print("\nANSWER:", result["answer"])
    print("\nSOURCES:", result["sources"])
    print("-" * 60)


QUESTION: What is preventive maintenance?

ANSWER: Preventive maintenance is a set of actions done on a regular schedule—by time or machine usage—to spot, stop, or lessen wear and tear on a component or system. The goal is to keep the equipment working well and extend its useful life by controlling degradation to an acceptable level. [S2][S3]

SOURCES: [{'id': 'S1', 'source': 'omguide_complete_w-eo-disclaimer.pdf', 'page': 51, 'score': 0.7068822070084066}, {'id': 'S2', 'source': 'omguide_complete_w-eo-disclaimer.pdf', 'page': 51, 'score': 0.7017281614717278}, {'id': 'S3', 'source': 'omguide_complete_w-eo-disclaimer.pdf', 'page': 51, 'score': 0.6884995158349395}]
------------------------------------------------------------

QUESTION: Why is machinery history important?

ANSWER: Machinery history is important because it lets you see how a machine has been performing over time.  
- By tracking past current and vibration signals you can spot changes that signal wear or damage before a fai

Test hallucination resistance

In [39]:
question = "Who invented the first computer?"

result = rag_answer(question)

print("QUESTION:", question)

print("\nFACTORYMIND AI ANSWER:")
print(result["answer"])

print("\nRETRIEVED SOURCES:")
for source in result["sources"]:
    print(source)

QUESTION: Who invented the first computer?

FACTORYMIND AI ANSWER:
The supplied context does not contain any information about who invented the first computer. [S1] discusses energy‑management systems, [S2] covers vibration equipment, and [S3] lists company contacts and resources – none of which mention the inventor of the first computer. Therefore, I cannot provide an answer based on the given material.

RETRIEVED SOURCES:
{'id': 'S1', 'source': 'omguide_complete_w-eo-disclaimer.pdf', 'page': 194, 'score': 0.20513401955365687}
{'id': 'S2', 'source': 'omguide_complete_w-eo-disclaimer.pdf', 'page': 84, 'score': 0.1861325114516664}
{'id': 'S3', 'source': 'omguide_complete_w-eo-disclaimer.pdf', 'page': 87, 'score': 0.16942985975410635}


RAG evaluation

In [42]:
evaluation_questions = [
    {
        "question": "What is preventive maintenance?",
        "expected": True
    },
    {
        "question": "Why is machinery history important?",
        "expected": True
    },
    {
        "question": "What are the benefits of predictive maintenance?",
        "expected": True
    },
    {
        "question": "Who invented the first computer?",
        "expected": False
    },
    {
        "question": "What is the capital of Japan?",
        "expected": False
    }
]

threshold = 0.25

for item in evaluation_questions:

    results = semantic_search(item["question"], top_k=3)

    relevant = [
        r for r in results
        if r.score >= threshold
    ]

    predicted = len(relevant) > 0

    print("\nQuestion:", item["question"])
    print("Expected answerable:", item["expected"])
    print("Retrieved chunks:", len(relevant))
    print("System found context:", predicted)
    print("Evaluation:", "PASS" if predicted == item["expected"] else "CHECK")
    print("-" * 60)


Question: What is preventive maintenance?
Expected answerable: True
Retrieved chunks: 3
System found context: True
Evaluation: PASS
------------------------------------------------------------

Question: Why is machinery history important?
Expected answerable: True
Retrieved chunks: 3
System found context: True
Evaluation: PASS
------------------------------------------------------------

Question: What are the benefits of predictive maintenance?
Expected answerable: True
Retrieved chunks: 3
System found context: True
Evaluation: PASS
------------------------------------------------------------

Question: Who invented the first computer?
Expected answerable: False
Retrieved chunks: 0
System found context: False
Evaluation: PASS
------------------------------------------------------------

Question: What is the capital of Japan?
Expected answerable: False
Retrieved chunks: 0
System found context: False
Evaluation: PASS
------------------------------------------------------------


In [43]:
passed = 0
total = len(evaluation_questions)

for item in evaluation_questions:

    results = semantic_search(item["question"], top_k=3)

    relevant = [
        r for r in results
        if r.score >= 0.25
    ]

    predicted = len(relevant) > 0
    expected = item["expected"]

    status = predicted == expected

    if status:
        passed += 1

    print(
        f"{'PASS' if status else 'FAIL'} | "
        f"{item['question']} | "
        f"Expected: {expected} | "
        f"Actual: {predicted}"
    )

print(f"\nFinal Score: {passed}/{total}")
print(f"Accuracy: {passed / total * 100:.1f}%")

PASS | What is preventive maintenance? | Expected: True | Actual: True
PASS | Why is machinery history important? | Expected: True | Actual: True
PASS | What are the benefits of predictive maintenance? | Expected: True | Actual: True
PASS | Who invented the first computer? | Expected: False | Actual: False
PASS | What is the capital of Japan? | Expected: False | Actual: False

Final Score: 5/5
Accuracy: 100.0%
